In [ ]:
# @title 📦 **نصب و راه‌اندازی**
# @markdown <div dir="rtl">
# @markdown ⏳ این مرحله حدود ۳ دقیقه طول می‌کشد.
# @markdown </div>

from IPython.display import clear_output
import os
import subprocess

colab_path = "/content"
kaggle_path = "/kaggle/working"

if os.path.exists(colab_path):
    print("🔵 در حال اتصال به Google Drive...")
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)
    path = "/content"
    print("✅ اتصال برقرار شد")
elif os.path.exists(kaggle_path):
    print("⚠️ محیط Kaggle شناسایی شد")
    path = "/kaggle/working"
else:
    raise EnvironmentError("❌ خطا: محیط اجرا شناسایی نشد")

try:
    print("\n📥 در حال نصب کتابخانه‌ها...")
    # نصب نسخه‌های سازگار بدون قفل روی نسخه‌های منقضی‌شده
    subprocess.run(
        ["pip", "install", "-q", "audio-separator[gpu]", "onnxruntime-gpu"],
        check=True,
    )
    subprocess.run(
        [
            "pip",
            "install",
            "-q",
            "demucs",
            "yt-dlp",
            "pydub",
            "tqdm",
            "aria2p",
        ],
        check=True,
    )

    print("🔧 در حال نصب ابزارهای سیستمی...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "ffmpeg", "aria2"], check=True
    )

    print("📂 در حال ایجاد پوشه‌ها...")
    os.makedirs("models", exist_ok=True)
    os.makedirs("temp", exist_ok=True)

    print("⬇️ در حال دانلود مدل DrumSep...")
    result = subprocess.run(
        [
            "aria2c",
            "https://huggingface.co/Eddycrack864/Drumsep/resolve/main/modelo_final.th",
            "-o",
            "models/drumsep.th",
            "--quiet=true",
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print("⚠️ هشدار: دانلود مدل DrumSep با مشکل مواجه شد")

    clear_output()
    print("✅ نصب با موفقیت انجام شد!")
    print("🚀 آماده برای پردازش")

except subprocess.CalledProcessError as e:
    print(f"❌ خطا در نصب: {str(e)}")
    raise
except Exception as e:
    print(f"❌ خطای غیرمنتظره: {str(e)}")
    raise

In [ ]:
# @title 🎵 **جداسازی صدا (Audio Separation)**

import glob
import os
import subprocess
import zipfile
from google.colab import files
from IPython.display import clear_output
from pydub import AudioSegment, silence
from tqdm import tqdm
import yt_dlp

# @markdown ### 🎯 **انتخاب هدف کاری**
User_Goal = "1. ساخت مدل هوش مصنوعی (وکال کریستالی)"  # @param ["1. ساخت مدل هوش مصنوعی (وکال کریستالی)", "2. بیت‌ساز و ریمیکسر (تفکیک 4 لاین)", "3. نوازنده (گیتار و پیانو)", "4. یوتیوبر/تولید محتوا (حذف نویز و اکو)", "5. دی‌جی/کارائوکه (موزیک بی‌کلام قوی)"]

# @markdown ---
# @markdown ### 📂 **انتخاب فایل ورودی**
Upload_From_Computer = False  # @param {type:"boolean"}
# @markdown <small>برای آپلود از حافظه داخلی، این گزینه را فعال کنید</small>

Audio_Input = ""  # @param {type:"string"}
# @markdown <small>مسیر فایل در درایو یا لینک یوتیوب/صوت</small>

# @markdown ---
# @markdown ### 💾 **تنظیمات خروجی**
Output_Folder = "/content/drive/MyDrive/UVR_Outputs"  # @param {type:"string"}
Output_Format = "wav"  # @param ["wav", "flac", "mp3"]
Download_As_Zip = False  # @param {type:"boolean"}

# @markdown ---
# @markdown ### ✂️ **پردازش نهایی**
Remove_Silence = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ### ⚙️ **تنظیمات دستی (اختیاری)**
Manual_Override = False  # @param {type:"boolean"}
Model_Select = "BS-Roformer-Viperx-1297"  # @param ["BS-Roformer-Viperx-1297", "MDX23C-8KFFT-InstVoc_HQ", "htdemucs_ft.yaml", "htdemucs_6s.yaml", "UVR-DeEcho-DeReverb.pth", "UVR-MDX-NET-Inst_HQ_5.onnx", "Mel-Band-Roformer-Kim"]

# ─────────────────────────────────────────────────────────────
# تعریف مدل‌ها
# ─────────────────────────────────────────────────────────────
MODEL_KZ = {
    "BS-Roformer-Viperx-1297": {
        "file": "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
        "arch": "mdxc",
        "seg": 256,
        "overlap": 8,
        "desc": "🎤 بهترین کیفیت برای جداسازی وکال - مناسب ساخت دیتاست AI",
    },
    "Mel-Band-Roformer-Kim": {
        "file": "mel_band_roformer_kim_ft2_bleedless_unwa.ckpt",
        "arch": "mdxc",
        "seg": 256,
        "overlap": 8,
        "desc": "🎵 وکال تمیز بدون نشت صدای موسیقی",
    },
    "MDX23C-8KFFT-InstVoc_HQ": {
        "file": "MDX23C-8KFFT-InstVoc_HQ.ckpt",
        "arch": "mdxc",
        "seg": 256,
        "overlap": 8,
        "desc": "🎹 جداسازی وکال/اینسترومنتال با کیفیت بالا",
    },
    "htdemucs_ft.yaml": {
        "file": "htdemucs_ft.yaml",
        "arch": "demucs",
        "shifts": 2,
        "overlap": 0.25,
        "desc": "🎸 تفکیک 4 ساز (درامز، بیس، وکال، سایر)",
    },
    "htdemucs_6s.yaml": {
        "file": "htdemucs_6s.yaml",
        "arch": "demucs",
        "shifts": 2,
        "overlap": 0.25,
        "desc": "🎹 تفکیک 6 ساز",
    },
    "UVR-DeEcho-DeReverb.pth": {
        "file": "UVR-DeEcho-DeReverb.pth",
        "arch": "vr",
        "window": 512,
        "aggr": 5,
        "desc": "🎙️ حذف اکو و ریورب از وکال",
    },
    "UVR-MDX-NET-Inst_HQ_5.onnx": {
        "file": "UVR-MDX-NET-Inst_HQ_5.onnx",
        "arch": "mdx",
        "seg": 256,
        "overlap": 0.25,
        "desc": "🎼 جداسازی سریع و با کیفیت وکال/موسیقی",
    },
}

path = "/content"


def validate_inputs():
    errors = []
    if not Upload_From_Computer and not Audio_Input:
        errors.append(
            "❌ لطفاً یک فایل آپلود کنید یا مسیر/لینک فایل را وارد کنید"
        )
    if (
        Audio_Input
        and not Upload_From_Computer
        and "http" not in Audio_Input
        and not os.path.exists(Audio_Input)
    ):
        errors.append(f"❌ فایل در مسیر '{Audio_Input}' یافت نشد")
    return errors


def get_settings_by_goal(goal):
    goal_map = {
        "1.": "BS-Roformer-Viperx-1297",
        "2.": "htdemucs_ft.yaml",
        "3.": "htdemucs_6s.yaml",
        "4.": "UVR-DeEcho-DeReverb.pth",
        "5.": "MDX23C-8KFFT-InstVoc_HQ",
    }
    for key, model in goal_map.items():
        if key in goal:
            return MODEL_KZ[model]
    return MODEL_KZ["BS-Roformer-Viperx-1297"]


def downloader(url):
    ydl_opts = {
        "format": "bestaudio/best",
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "wav",
            "preferredquality": "192",
        }],
        "outtmpl": os.path.join(f"{path}/temp", "%(title)s.%(ext)s"),
        "quiet": False,
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return True
    except Exception as e:
        print(f"❌ خطا در دانلود: {str(e)}")
        return False


def process_silence_removal(folder_path, format_ext):
    print(f"\n✂️ در حال پردازش حذف سکوت...")
    all_output_files = glob.glob(f"{folder_path}/*.{format_ext}")
    vocal_files = [
        f
        for f in all_output_files
        if "vocal" in os.path.basename(f).lower()
        and "_NoSilence" not in os.path.basename(f)
    ]

    for file_path in vocal_files:
        try:
            audio = AudioSegment.from_file(file_path)
            chunks = silence.split_on_silence(
                audio, min_silence_len=500, silence_thresh=-45, keep_silence=100
            )
            if chunks:
                output_audio = sum(chunks)
                base_name, ext = os.path.splitext(file_path)
                output_audio.export(f"{base_name}_NoSilence{ext}", format=format_ext)
                print(f"✅ سکوت از {os.path.basename(file_path)} حذف شد")
        except Exception as e:
            print(f"❌ خطا در حذف سکوت: {str(e)}")


def run_separation():
    validation_errors = validate_inputs()
    if validation_errors:
        for error in validation_errors:
            print(error)
        return

    # اصلاح مسیر خروجی اگر لینک یوتیوب داده شده باشد
    global Output_Folder
    if "http" in Output_Folder or "youtu" in Output_Folder:
        Output_Folder = "/content/drive/MyDrive/UVR_Outputs"
        print(
            f"⚠️ مسیر خروجی نامعتبر بود؛ به‌صورت خودکار تغییر یافت به:"
            f" {Output_Folder}"
        )

    current_settings = (
        MODEL_KZ[Model_Select]
        if Manual_Override
        else get_settings_by_goal(User_Goal)
    )
    print(f"📌 مدل انتخابی: {current_settings['file']}")

    os.makedirs(f"{path}/temp", exist_ok=True)
    for f in glob.glob(f"{path}/temp/*"):
        try:
            os.remove(f)
        except:
            pass

    input_path = Audio_Input
    if Upload_From_Computer:
        uploaded = files.upload()
        if not uploaded:
            return
        for filename in uploaded.keys():
            os.rename(
                os.path.join(os.getcwd(), filename),
                os.path.join(f"{path}/temp", filename),
            )
        input_path = f"{path}/temp"
    elif "http" in Audio_Input:
        print("\n⬇️ در حال دانلود ویدیو/صوت...")
        if not downloader(Audio_Input):
            return
        input_path = f"{path}/temp"

    found_files = []
    extensions = (".wav", ".flac", ".mp3", ".ogg", ".m4a")
    if os.path.isdir(input_path):
        for f in os.listdir(input_path):
            if f.lower().endswith(extensions):
                found_files.append(os.path.join(input_path, f))
    elif input_path.lower().endswith(extensions):
        found_files.append(input_path)

    if not found_files:
        print("❌ هیچ فایل صوتی پیدا نشد!")
        return

    os.makedirs(Output_Folder, exist_ok=True)
    model_file = current_settings["file"]
    arch = current_settings["arch"]

    for idx, file_path in enumerate(found_files, 1):
        print(
            f"\n🚀 شروع پردازش فایل {idx}/{len(found_files)}:"
            f" {os.path.basename(file_path)}"
        )

        base_cmd = f'audio-separator "{file_path}" --model_filename {model_file} --output_dir="{Output_Folder}" --output_format={Output_Format} --model_file_dir=./models'

        if arch == "mdxc":
            cmd = f'{base_cmd} --mdxc_segment_size={current_settings["seg"]} --mdxc_overlap={current_settings["overlap"]}'
        elif arch == "demucs":
            cmd = f'{base_cmd} --demucs_shifts={current_settings["shifts"]} --demucs_overlap={current_settings["overlap"]}'
        elif arch == "vr":
            cmd = f'{base_cmd} --vr_window_size={current_settings["window"]} --vr_aggression={current_settings["aggr"]}'
        else:
            cmd = base_cmd

        # اجرا به‌صورت زنده برای دیدن پیشرفت
        process = subprocess.Popen(
            cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        for line in process.stdout:
            print(line, end="")
        process.wait()

    if Remove_Silence:
        process_silence_removal(Output_Folder, Output_Format)

    print("\n🎉 پردازش با موفقیت تمام شد!")
    print(f"📂 پوشه خروجی: {Output_Folder}")


try:
    run_separation()
except Exception as e:
    print(f"\n❌ خطای کلی: {str(e)}")

In [ ]:

#@title 📂 **مدیریت هوشمند فایل (نسخه موبایل و دسکتاپ)**

import os
import html
from IPython.display import HTML, display

#@markdown مسیر پوشه خروجی را وارد کنید:
Target_Folder = "/content/drive/MyDrive/UVR_Outputs" #@param {type:"string"}

def list_refined_files_v2(folder_path):
    if not os.path.exists(folder_path):
        display(HTML(f"<div style='color:red; padding:10px;'>❌ مسیر یافت نشد: {folder_path}</div>"))
        return

    files_found = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.wav', '.mp3', '.flac')):
                if "Instrumental" in file: continue
                if "(Reverb)" in file and "(No Reverb)" not in file: continue

                full_path = os.path.join(root, file)
                files_found.append((file, full_path))

    if not files_found:
        print("⚠️ فایلی پیدا نشد.")
        return

    # مرتب‌سازی: No Reverb بالاتر از همه
    files_found.sort(key=lambda x: "No Reverb" in x[0], reverse=True)

    html_code = """
    <style>
        .container { font-family: sans-serif; direction: rtl; background: #1a1a1a; padding: 10px; border-radius: 10px; color: white; }
        .file-card {
            background: #252525;
            border-radius: 8px;
            padding: 15px;
            margin-bottom: 15px;
            border-right: 4px solid #00d2ff;
            display: flex;
            flex-direction: column; /* چیدمان عمودی برای موبایل */
            gap: 10px;
        }
        .tag {
            display: inline-block;
            padding: 4px 10px;
            border-radius: 5px;
            font-size: 12px;
            font-weight: bold;
            margin-bottom: 5px;
        }
        .tag-crystal { background: #00d2ff; color: #000; }
        .tag-vocal { background: #4caf50; color: white; }

        .file-name {
            font-size: 14px;
            word-break: break-all; /* شکستن کلمات طولانی */
            direction: ltr;
            text-align: left;
            color: #efefef;
            line-height: 1.4;
        }
        .copy-btn {
            background: #007bff;
            color: white;
            border: none;
            padding: 12px;
            border-radius: 6px;
            cursor: pointer;
            font-size: 14px;
            font-weight: bold;
            width: 100%; /* دکمه تمام عرض برای راحتی لمس در موبایل */
            margin-top: 5px;
        }
    </style>
    <script>
        function copyText(text, btnId) {
            var textArea = document.createElement("textarea");
            textArea.value = text;
            document.body.appendChild(textArea);
            textArea.select();
            try {
                document.execCommand('copy');
                var btn = document.getElementById(btnId);
                btn.innerText = "✅ کپی شد!";
                btn.style.background = "#28a745";
                setTimeout(function(){
                    btn.innerText = "📋 کپی مسیر فایل";
                    btn.style.background = "#007bff";
                }, 2000);
            } catch (err) {
                alert('خطا در کپی');
            }
            document.body.removeChild(textArea);
        }
    </script>
    <div class="container">
        <h3 style="text-align: center; color: #00d2ff;">✨ لیست فایل‌های نهایی</h3>
    """

    for i, (name, full_path) in enumerate(files_found):
        btn_id = f"btn_new_{i}"

        # انتخاب تگ
        if "No Reverb" in name:
            tag = '<span class="tag tag-crystal">💎 Crystal Clear</span>'
            border = "border-right: 4px solid #00d2ff;"
        elif "NoSilence" in name:
            tag = '<span class="tag tag-vocal">🎙️ No Silence</span>'
            border = "border-right: 4px solid #4caf50;"
        else:
            tag = '<span class="tag" style="background:#666;">🎵 Raw</span>'
            border = "border-right: 4px solid #888;"

        html_code += f"""
        <div class="file-card" style="{border}">
            <div>{tag}</div>
            <div class="file-name">{name}</div>
            <button id="{btn_id}" class="copy-btn" onclick="copyText('{html.escape(full_path)}', '{btn_id}')">📋 کپی مسیر فایل</button>
        </div>
        """

    html_code += "</div>"
    display(HTML(html_code))

list_refined_files_v2(Target_Folder)

In [ ]:

#@title 🔊 **نرمالایز LUFS + انتخاب هوشمند کانال (Mono-Safe) + خروجی FLAC**

!pip install pyloudnorm soundfile -q

import os
import glob
import numpy as np
import soundfile as sf
import pyloudnorm as pyln
from tqdm import tqdm

Input_Folder = "/content/drive/MyDrive/UVR_Outputs" #@param {type:"string"}
Target_LUFS = -20.0 #@param {type:"number"}
True_Peak_Limit_dB = -1.0 #@param {type:"number"}
Download_Normalized_As_Zip = True #@param {type:"boolean"}

def check_crystal_vocals():
    if not os.path.exists(Input_Folder):
        return False, []
    extensions = ('*.wav', '*.flac', '*.mp3')
    all_files = []
    for ext in extensions:
        all_files.extend(glob.glob(os.path.join(Input_Folder, ext)))
    crystal_files = []
    for f in all_files:
        filename = os.path.basename(f)
        if "NoSilence" in filename and "(No Reverb)" in filename:
            if "_Normalized" not in filename:
                crystal_files.append(f)
    return len(crystal_files) > 0, crystal_files

def pick_best_channel(data):
    """
    به‌جای mix کردن L+R (که باعث phase cancellation و صدای بادمانند می‌شه)،
    کانالی که انرژی/وضوح بیشتری داره رو انتخاب می‌کنه.
    """
    if data.ndim == 1:
        return data  # از قبل مونو هست

    left = data[:, 0]
    right = data[:, 1]

    # معیار انتخاب: RMS energy - کانالی که سیگنال قوی‌تر/واضح‌تری داره
    rms_left = np.sqrt(np.mean(left**2))
    rms_right = np.sqrt(np.mean(right**2))

    if rms_left >= rms_right:
        return left
    else:
        return right

def run_lufs_normalization():
    print("🔍 در حال جستجوی فایل‌های واجد شرایط...")
    has_files, target_files = check_crystal_vocals()

    if not has_files:
        print("⚠️ فایلی با مشخصات NoSilence + No Reverb پیدا نشد.")
        return

    success_files = []
    for file_path in tqdm(target_files, desc="Processing"):
        try:
            data, rate = sf.read(file_path)

            # مرحله ۱: انتخاب بهترین کانال (به‌جای mix خودکار استریو→مونو)
            mono_data = pick_best_channel(data)

            # مرحله ۲: نرمالایز LUFS
            meter = pyln.Meter(rate)
            current_loudness = meter.integrated_loudness(mono_data)
            normalized_audio = pyln.normalize.loudness(mono_data, current_loudness, Target_LUFS)

            # مرحله ۳: جلوگیری از کلیپ (true peak protection)
            peak = np.max(np.abs(normalized_audio))
            peak_dB = 20 * np.log10(peak) if peak > 0 else -np.inf
            if peak_dB > True_Peak_Limit_dB:
                gain_reduction = 10 ** ((True_Peak_Limit_dB - peak_dB) / 20)
                normalized_audio = normalized_audio * gain_reduction

            base, _ = os.path.splitext(file_path)
            new_path = f"{base}_Normalized.flac"

            sf.write(new_path, normalized_audio, rate, format='FLAC')

            if os.path.exists(new_path):
                success_files.append(new_path)
        except Exception as e:
            print(f"❌ Error in {os.path.basename(file_path)}: {e}")

    print(f"\n{'='*50}")
    if success_files:
        print(f"✅ موفقیت‌آمیز: {len(success_files)} فایل مونو، نرمالایز و فشرده شد (FLAC).")
        for p in success_files: print(f"📍 Created: {os.path.basename(p)}")
    else:
        print("❌ هیچ فایلی ذخیره نشد. دسترسی به درایو را چک کنید.")
    print(f"{'='*50}")

    if Download_Normalized_As_Zip and success_files:
        import zipfile
        from google.colab import files as colab_files
        z_path = "/content/Normalized_Result.zip"
        with zipfile.ZipFile(z_path, 'w', zipfile.ZIP_DEFLATED) as z:
            for f in success_files: z.write(f, os.path.basename(f))
        colab_files.download(z_path)

run_lufs_normalization()

###برای نرمالایز صدا توسط اپلیکیشن ، این ویدیو را تماشا کنید
[![آموزش ویدیویی](https://img.youtube.com/vi/vxi_7frcXhY/0.jpg)](https://youtu.be/vxi_7frcXhY?si=wP5xfoHvw1tU28H2)

[🎥 مشاهده ویدیو در یوتیوب](https://youtu.be/vxi_7frcXhY?si=LngLfdr7sAeekA6Y)